Refer this link to understand deeply: 
https://www.geeksforgeeks.org/deep-learning/bidirectional-recurrent-neural-network/

Overview of Bidirectional Recurrent Neural Networks (BRNNs)
A Bidirectional Recurrent Neural Network (BRNN) is an extension of the traditional RNN that processes sequential data in both forward and backward directions. This allows the network to utilize both past and future context when making predictions providing a more comprehensive understanding of the sequence.

Like a traditional RNN, a BRNN moves forward through the sequence, updating the hidden state based on the current input and the prior hidden state at each time step. The key difference is that a BRNN also has a backward hidden layer which processes the sequence in reverse, updating the hidden state based on the current input and the hidden state of the next time step.

Compared to unidirectional RNNs BRNNs improve accuracy by considering both the past and future context. This is because the two hidden layers i.e forward and backward complement each other and predictions are made using the combined outputs of both layers.

Example:
Consider the sentence: "I like apple. It is very healthy."

one layer more deeper into architecture with equations follow this : 
https://medium.com/@fraidoonomarzai99/bidirectional-rnn-in-depth-1efd32c3cf46

In [2]:
import warnings
warnings.filterwarnings('ignore')
from keras.datasets import imdb
from keras.preprocessing.sequence import pad_sequences

features = 2000  # Number of most frequent words to consider
max_len = 50     # Maximum length of each sequence

(X_train, y_train), (X_test, y_test) = imdb.load_data(num_words=features)

X_train = pad_sequences(X_train, maxlen=max_len)
X_test = pad_sequences(X_test, maxlen=max_len)

In [3]:
from keras.models import Sequential
from keras.layers import Embedding, Bidirectional, SimpleRNN, Dense

embedding_dim = 128  
hidden_units = 64    

model = Sequential()

model.add(Embedding(features, embedding_dim, input_length=max_len))

model.add(Bidirectional(SimpleRNN(hidden_units)))

model.add(Dense(1, activation='sigmoid'))

model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

In [4]:
batch_size = 32
epochs = 20

model.fit(X_train, y_train,
          batch_size=batch_size,
          epochs=epochs,
          validation_data=(X_test, y_test))

Epoch 1/20
782/782 [==============================] - 8s 10ms/step - loss: 0.5345 - accuracy: 0.7262 - val_loss: 0.4677 - val_accuracy: 0.7793
Epoch 2/20
782/782 [==============================] - 8s 10ms/step - loss: 0.4290 - accuracy: 0.8049 - val_loss: 0.4592 - val_accuracy: 0.7870
Epoch 3/20
782/782 [==============================] - 8s 11ms/step - loss: 0.3513 - accuracy: 0.8464 - val_loss: 0.5025 - val_accuracy: 0.7800
Epoch 4/20
782/782 [==============================] - 8s 11ms/step - loss: 0.2452 - accuracy: 0.9012 - val_loss: 0.5909 - val_accuracy: 0.7750
Epoch 5/20
782/782 [==============================] - 8s 11ms/step - loss: 0.1583 - accuracy: 0.9402 - val_loss: 0.7489 - val_accuracy: 0.7376
Epoch 6/20
782/782 [==============================] - 8s 11ms/step - loss: 0.1004 - accuracy: 0.9628 - val_loss: 0.9003 - val_accuracy: 0.7335
Epoch 7/20
782/782 [==============================] - 8s 11ms/step - loss: 0.0675 - accuracy: 0.9762 - val_loss: 1.1149 - val_accuracy: 0.7400

In [15]:
loss, accuracy = model.evaluate(X_test, y_test)

print('Test accuracy:', accuracy)

782/782 [==============================] - 2s 2ms/step - loss: 1.6774 - accuracy: 0.7288
Test accuracy: 0.7287600040435791


In [16]:
from sklearn.metrics import classification_report

y_pred = model.predict(X_test)

y_pred = (y_pred > 0.5)

print(classification_report(y_test, y_pred, target_names=['Negative', 'Positive']))

782/782 [==============================] - 2s 2ms/step
              precision    recall  f1-score   support

    Negative       0.75      0.69      0.72     12500
    Positive       0.71      0.77      0.74     12500

    accuracy                           0.73     25000
   macro avg       0.73      0.73      0.73     25000
weighted avg       0.73      0.73      0.73     25000



In [1]:
# lets test on a custom review
custom_review = "This movie was good."

# Use the same IMDB word index mapping as training
from keras.datasets import imdb
word_index = imdb.get_word_index()

# Encode raw text using IMDB word index, with +3 shift for reserved indices
def encode_text(text, word_index, max_words):
    tokens = text.lower().replace('.', '').replace(',', '').split()
    encoded = []
    for word in tokens:
        if word in word_index:
            idx = word_index[word] + 3
            if idx < max_words:
                encoded.append(idx)
    return encoded

custom_encoded = encode_text(custom_review, word_index, features)
custom_padded = pad_sequences([custom_encoded], maxlen=max_len)
custom_prediction = model.predict(custom_padded)
print('Custom review prediction (1=Positive, 0=Negative):', (custom_prediction > 0.5).astype(int)[0][0])
print('Raw probability:', float(custom_prediction[0][0]))

NameError: name 'features' is not defined